# Jak komputer czyta tekst - od liczenia słów do wektorów

Ten notatnik to companion do blog posta: [Jak komputer czyta tekst](jak-komputer-czyta-tekst.html)

**Instalacja:**
```
pip install tiktoken transformers tokenizers scikit-learn pandas gensim numpy matplotlib seaborn
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

---
## 1. Tokenizacja

### Tiktoken - jak GPT tnie tekst

In [ ]:
import tiktoken

text = "Nieprawdopodobnie szczęśliwy"

gpt2 = tiktoken.get_encoding("gpt2")
gpt4 = tiktoken.get_encoding("cl100k_base")

print("GPT-2:", gpt2.encode(text))
print("GPT-4:", gpt4.encode(text))

print("GPT-2:", [gpt2.decode([t]) for t in gpt2.encode(text)])
print("GPT-4:", [gpt4.decode([t]) for t in gpt4.encode(text)])

### Polskie tokenizatory

In [ ]:
from transformers import AutoTokenizer

text = "Nieprawdopodobnie szczęśliwy"

herbert = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
print("HerBERT (Allegro, polski WordPiece):", herbert.tokenize(text))

roberta = AutoTokenizer.from_pretrained("sdadas/polish-roberta-base-v2")
print("Polish RoBERTa (polski SentencePiece):", roberta.tokenize(text))

### Wizualizacja: porównanie liczby tokenów

In [ ]:
test_texts = [
    "Nieprawdopodobnie szczęśliwy",
    "Kot siedzi na macie i śpi",
    "Szczęście to stan umysłu",
]

results = {}
for text in test_texts:
    results[text] = {
        "GPT-2": len(gpt2.encode(text)),
        "GPT-4o": len(gpt4.encode(text)),
        "HerBERT": len(herbert.tokenize(text)),
        "Polish RoBERTa": len(roberta.tokenize(text)),
    }

df_tokens = pd.DataFrame(results).T
display(df_tokens)

fig, ax = plt.subplots(figsize=(10, 4))
df_tokens.plot(kind="barh", ax=ax, colormap="Set2")
ax.set_xlabel("Liczba tokenów")
ax.set_title("Porównanie tokenizatorów na polskim tekście")
ax.legend(title="Tokenizer")
for container in ax.containers:
    ax.bar_label(container, fontsize=8)
plt.tight_layout()
plt.show()

Zauważ: GPT-2 potrzebuje najwięcej tokenów na polski tekst, a Polish RoBERTa najmniej. To dlatego RoBERTa był trenowany specjalnie na polskim.

### Własny tokenizer BPE

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(vocab_size=300, special_tokens=["[UNK]"])

corpus = [
    "Ala ma kota i psa",
    "Kot siedzi na macie",
    "Pies biega po parku",
    "Ala ma kota i kota i jeszcze raz kota",
    "Kot lubi mleko i kot lubi spać",
    "Pies lubi biegać po parku i gonić kota",
    "Szczęśliwy kot to kot który ma dużo mleka",
    "Nieprawdopodobnie szczęśliwy pies biega po parku",
    "Szczęśliwy Ala ma kota i psa i szczęśliwy dom",
    "Dom to miejsce gdzie mieszka szczęśliwa rodzina",
]

tokenizer.train_from_iterator(corpus, trainer)
print(f"Rozmiar słownika: {tokenizer.get_vocab_size()}")

test = "Nieprawdopodobnie szczęśliwy"
encoding = tokenizer.encode(test)
print(f'"{test}" → {encoding.tokens}')

---
## 2. BoW i TF-IDF

### Bag of Words (BoW)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "Ten film jest świetny",
    "Ten film jest beznadziejny",
    "Świetny film polecam",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

df_bow = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
display(df_bow)

fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(df_bow, annot=True, cmap="Blues", ax=ax, cbar_kws={"label": "Liczba wystąpień"})
ax.set_title("Bag of Words - zliczenia słów")
plt.tight_layout()
plt.show()

### TF-IDF z heatmapą

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
display(df_tfidf.round(2))

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(df_tfidf, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "TF-IDF"})
ax.set_title("TF-IDF - ważność słów w dokumentach")
plt.tight_layout()
plt.show()

Porównaj BoW i TF-IDF: "film" jest wszędzie w BoW (wartość 1), ale w TF-IDF ma niskie wartości - bo nie wnosi informacji pozwalającej odróżnić dokumenty.

### TF-IDF jako wyszukiwarka

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query = "świetny film"
query_vec = vectorizer.transform([query])

similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
ranked = sorted(zip(corpus, similarities), key=lambda x: -x[1])
for i, (doc, sim) in enumerate(ranked, 1):
    print(f'{i}. "{doc}" → {sim:.3f}')

fig, ax = plt.subplots(figsize=(8, 3))
docs_short = [f"Dok {i+1}" for i in range(len(corpus))]
colors = ["#2ecc71" if s == max(similarities) else "#bdc3c7" for s in similarities]
ax.barh(docs_short, similarities, color=colors)
ax.set_xlabel("Podobieństwo kosinusowe")
ax.set_title(f'Wyniki wyszukiwania dla: "{query}"')
for i, s in enumerate(similarities):
    ax.text(s + 0.01, i, f'{s:.3f}', va='center')
plt.tight_layout()
plt.show()

### N-gramy z TF-IDF

In [ ]:
vectorizer_ngram = TfidfVectorizer(ngram_range=(1, 2))
tfidf_ngram = vectorizer_ngram.fit_transform(corpus)

df_ngram = pd.DataFrame(
    tfidf_ngram.toarray(),
    columns=vectorizer_ngram.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)

fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(df_ngram, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "TF-IDF"})
ax.set_title("N-gramy (1,2) z TF-IDF - unigramy + bigramy")
plt.tight_layout()
plt.show()

---
## 3. Łańcuchy Markowa - generator tekstu

In [ ]:
import random

corpus_markov = "ala ma kota ala ma psa kot lubi mleko ala lubi kota"
tokens = corpus_markov.split()

trigrams = {}
for i in range(len(tokens) - 3):
    key = (tokens[i], tokens[i + 1], tokens[i + 2])
    next_word = tokens[i + 3]
    if key not in trigrams:
        trigrams[key] = []
    trigrams[key].append(next_word)

current = ("ala", "ma", "kota")
output = list(current)

for _ in range(10):
    if tuple(output[-3:]) not in trigrams:
        break
    possibilities = trigrams[tuple(output[-3:])]
    output.append(random.choice(possibilities))

print(" ".join(output))

### Wizualizacja: graf przejść Markowa

In [ ]:
from collections import defaultdict, Counter

transitions = defaultdict(Counter)
for i in range(len(tokens) - 1):
    transitions[tokens[i]][tokens[i + 1]] += 1

fig, ax = plt.subplots(figsize=(8, 4))
all_words = sorted(set(tokens))

transition_matrix = np.zeros((len(all_words), len(all_words)))
word2i = {w: i for i, w in enumerate(all_words)}

for word, nexts in transitions.items():
    total = sum(nexts.values())
    for n, count in nexts.items():
        transition_matrix[word2i[word]][word2i[n]] = count / total

sns.heatmap(
    transition_matrix,
    xticklabels=all_words,
    yticklabels=all_words,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "P(przejście)"},
)
ax.set_xlabel("Następne słowo")
ax.set_ylabel("Obecne słowo")
ax.set_title("Macierz przejść - łańcuch Markowa")
plt.tight_layout()
plt.show()

---
## 4. Naive Bayes - klasyfikator spamu

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

emails = [
    "Kup viagra tanio teraz",
    "Viagra darmowa oferta",
    "Spotkanie jutro o 10",
    "Prześlij mi raport jutro",
    "Tanie leki online kup teraz",
    "Raport z wczoraj prześlij",
]
labels = [1, 1, 0, 0, 1, 0]  # 1=spam, 0=nie-spam

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails)

classifier = MultinomialNB(alpha=1.0)
classifier.fit(X, labels)

new_email = vectorizer.transform(["Viagra jutro spotkanie"])
prediction = classifier.predict(new_email)
probs = classifier.predict_proba(new_email)[0]
print(f"Klasyfikacja: {'Spam' if prediction[0] == 1 else 'Nie-spam'}")
print(f"P(spam) = {probs[1]:.3f}, P(nie-spam) = {probs[0]:.3f}")

### Wizualizacja: prawdopodobieństwa spam/nie-spam dla słów

In [ ]:
feature_names = vectorizer.get_feature_names_out()
log_probs_spam = classifier.feature_log_prob_[1]
log_probs_ham = classifier.feature_log_prob_[0]

df_probs = pd.DataFrame({
    "słowo": feature_names,
    "P(słowo|spam)": np.exp(log_probs_spam),
    "P(słowo|nie-spam)": np.exp(log_probs_ham),
}).sort_values("P(słowo|spam)", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df_probs))
width = 0.35
ax.bar([i - width/2 for i in x], df_probs["P(słowo|spam)"], width, label="Spam", color="#e74c3c", alpha=0.8)
ax.bar([i + width/2 for i in x], df_probs["P(słowo|nie-spam)"], width, label="Nie-spam", color="#2ecc71", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_probs["słowo"], rotation=45, ha="right")
ax.set_ylabel("Prawdopodobieństwo")
ax.set_title("P(słowo | klasa) - które słowa wskazują na spam?")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Word2Vec - embeddingi od zera

### CBOW od zera (czysty numpy)

In [ ]:
import numpy as np

np.random.seed(42)

corpus = "kot siedzi na macie i śpi pies siedzi na dywanie i śpi kot biega po pokoju pies biega po parku".split()
vocab = list(set(corpus))
word2idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)

def one_hot(word):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec

print(f'Słownik ({vocab_size} słów): {vocab}')
print(f'One-hot "kot": {one_hot("kot")}')

In [ ]:
embed_dim = 5
W1 = np.random.randn(vocab_size, embed_dim) * 0.01
W2 = np.random.randn(embed_dim, vocab_size) * 0.01

window = 2
pairs = []
for i in range(window, len(corpus) - window):
    context = corpus[i - window:i] + corpus[i + 1:i + window + 1]
    target = corpus[i]
    pairs.append((context, target))

print(f'Liczba par treningowych: {len(pairs)}')
print(f'Przykładowa para: kontekst={pairs[0][0]} → cel={pairs[0][1]}')

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

losses = []
lr = 0.05
for epoch in range(500):
    loss = 0
    for context_words, target_word in pairs:
        context_idx = [word2idx[w] for w in context_words]
        target_idx = word2idx[target_word]

        hidden = np.mean(W1[context_idx], axis=0)
        output = softmax(hidden @ W2)
        loss -= np.log(output[target_idx] + 1e-8)

        grad = output.copy()
        grad[target_idx] -= 1
        W2 -= lr * np.outer(hidden, grad)
        grad_hidden = grad @ W2.T
        for idx in context_idx:
            W1[idx] -= lr * grad_hidden / len(context_idx)

    losses.append(loss)
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}, loss: {loss:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color="#3498db", linewidth=1)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (cross-entropy)")
ax.set_title("Trening CBOW - jak spada loss")
ax.set_xlim(0, len(losses))
plt.tight_layout()
plt.show()

for word in ["kot", "pies", "siedzi", "biega"]:
    print(f"{word}: {W1[word2idx[word]].round(3)}")

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print(f'\nkot ↔ pies: {cosine(W1[word2idx["kot"]], W1[word2idx["pies"]]):.3f}')
print(f'kot ↔ siedzi: {cosine(W1[word2idx["kot"]], W1[word2idx["siedzi"]]):.3f}')
print(f'kot ↔ biega: {cosine(W1[word2idx["kot"]], W1[word2idx["biega"]]):.3f}')

### Word2Vec z gensim + wizualizacja embeddingów (PCA)

In [ ]:
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

w2v_corpus = [
    ["kot", "siedzi", "na", "macie", "i", "śpi"],
    ["pies", "siedzi", "na", "dywanie", "i", "śpi"],
    ["kot", "biega", "po", "pokoju"],
    ["pies", "biega", "po", "parku"],
    ["kot", "i", "pies", "bawią", "się", "w", "ogrodzie"],
    ["kot", "je", "karmę", "z", "miski"],
    ["pies", "je", "karmę", "z", "miski"],
    ["kot", "łapie", "mysz", "w", "domu"],
    ["pies", "goni", "kota", "po", "ogrodzie"],
    ["ptak", "siedzi", "na", "gałęzi", "drzewa"],
    ["ryba", "pływa", "w", "akwarium"],
    ["samochód", "jeździ", "po", "drodze"],
    ["rower", "jeździ", "po", "ścieżce"],
    ["kot", "mruczy", "gdy", "głaszczesz", "go"],
    ["pies", "merda", "ogonem", "gdy", "widzi", "pana"],
]

model = Word2Vec(
    sentences=w2v_corpus,
    vector_size=10,
    window=3,
    min_count=1,
    sg=0,
    epochs=200,
)

print(model.wv.most_similar("kot", topn=3))
print(f'kot ↔ pies: {model.wv.similarity("kot", "pies"):.3f}')
print(f'kot ↔ samochód: {model.wv.similarity("kot", "samochód"):.3f}')

In [ ]:
words = list(model.wv.key_to_index.keys())
vectors = np.array([model.wv[w] for w in words])

pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.7, edgecolors="black", linewidth=0.5)

for i, word in enumerate(words):
    ax.annotate(word, (coords[i, 0] + 0.01, coords[i, 1] + 0.01), fontsize=9)

ax.set_title(f"Word2Vec embeddingi wizualizowane przez PCA (variance explained: {pca.explained_variance_ratio_.sum():.1%})")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
plt.tight_layout()
plt.show()

Mały korpus = brak wyraźnych klastrów. Ale na dużym korpusie (np. polska Wikipedia) "kot" i "pies" byłyby blisko, a "samochód" daleko.

### Heatmapa podobieństw między słowami

In [ ]:
selected = ["kot", "pies", "ptak", "ryba", "samochód", "rower", "siedzi", "biega", "je", "śpi"]
selected = [w for w in selected if w in model.wv]

sim_matrix = np.zeros((len(selected), len(selected)))
for i, w1 in enumerate(selected):
    for j, w2 in enumerate(selected):
        sim_matrix[i][j] = model.wv.similarity(w1, w2)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    sim_matrix,
    xticklabels=selected,
    yticklabels=selected,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "Podobieństwo kosinusowe"},
)
ax.set_title("Podobieństwo kosinusowe między słowami (Word2Vec)")
plt.tight_layout()
plt.show()

### Word2Vec + łańcuchy Markowa = generator z embeddingami

In [ ]:
from collections import defaultdict
import random

corpus_gen = [
    "kot siedzi na macie i śpi",
    "pies siedzi na dywanie i śpi",
    "kot biega po pokoju",
    "pies biega po parku",
    "kot i pies bawią się w ogrodzie",
    "kot je karmę z miski",
    "pies je karmę z miski",
    "kot łapie mysz w domu",
    "pies goni kota po ogrodzie",
    "ptak siedzi na gałęzi drzewa",
    "kot patrzy przez okno i mruczy",
    "pies szczeka na listonosza",
    "kot śpi cały dzień na kanapie",
]

tokenized = [sentence.split() for sentence in corpus_gen]

w2v_model = Word2Vec(sentences=tokenized, vector_size=10, window=3, min_count=1, epochs=200)

transitions = defaultdict(list)
for sentence in tokenized:
    for i in range(len(sentence) - 1):
        transitions[sentence[i]].append(sentence[i + 1])

def generate(start, length=6):
    words = [start]
    for _ in range(length):
        current = words[-1]
        if current not in transitions:
            break

        candidates = transitions[current]
        context_vector = np.mean(
            [w2v_model.wv[w] for w in words[-3:] if w in w2v_model.wv],
            axis=0,
        )

        scored = []
        for candidate in candidates:
            if candidate in w2v_model.wv:
                similarity = np.dot(context_vector, w2v_model.wv[candidate]) / (
                    np.linalg.norm(context_vector) * np.linalg.norm(w2v_model.wv[candidate]) + 1e-8
                )
                scored.append((candidate, similarity))

        if scored:
            weights = [score + 1 for _, score in scored]
            chosen = random.choices([w for w, _ in scored], weights=weights, k=1)[0]
            words.append(chosen)
        else:
            words.append(random.choice(candidates))
    return " ".join(words)

print("=== Wygenerowane teksty (kot) ===")
for _ in range(5):
    print(f"  {generate('kot')}")